# 00 - Setup: GLiNER + PyMuPDF

## 1. GLiNER basic usage

In [1]:
from gliner import GLiNER

model = GLiNER.from_pretrained("gliner-community/gliner_small-v2.5")

/home/edgherand/PycharmProjects/ner_project_EdgarHerranz/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/edgherand/PycharmProjects/ner_project_EdgarHerranz/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [2]:
text = "Cristiano Ronaldo was born on 5 February 1985 in Funchal. He won the Ballon d'Or and played for Real Madrid."
labels = ["person", "award", "date", "teams"]

entities = model.predict_entities(text, labels, threshold=0.5)
for e in entities:
    print(e["text"], "=>", e["label"])

Cristiano Ronaldo => person
5 February 1985 => date
Ballon d'Or => award
Real Madrid => teams


## 2. Structure returned by `predict_entities`

In [3]:
entities[0]

{'start': 0,
 'end': 17,
 'text': 'Cristiano Ronaldo',
 'label': 'person',
 'score': 0.9978237152099609}

In [4]:
for e in entities:
    print(e["start"], e["end"], repr(text[e["start"]:e["end"]]), e["label"], round(e["score"], 2))

0 17 'Cristiano Ronaldo' person 1.0
30 45 '5 February 1985' date 1.0
69 80 "Ballon d'Or" award 0.99
96 107 'Real Madrid' teams 1.0


**What are the spans?** `start` and `end` are the character positions of the entity inside the original text (`start` included, `end` excluded). So `text[start:end]` is exactly the detected entity. They tell you *where* the entity is, not just what it is (this is what strict evaluation compares).

The other fields: `text` = entity string, `label` = predicted label, `score` = model confidence.

## 3. Other texts and labels

In [5]:
text2 = "Maria Lopez lives at Carrer de Mallorca 12, Barcelona and paid 350 euros on 2024-03-10 to Acme S.L."
labels2 = ["person", "address", "amount", "date", "organization"]
for e in model.predict_entities(text2, labels2, threshold=0.5):
    print(e["start"], e["end"], e["text"], "=>", e["label"])

0 11 Maria Lopez => person
21 42 Carrer de Mallorca 12 => address
63 72 350 euros => amount
76 86 2024-03-10 => date
90 99 Acme S.L. => organization


## 4. PyMuPDF quick start

In [6]:
import pymupdf

# create a small sample PDF so this runs without any file
doc = pymupdf.open()
page = doc.new_page()
page.insert_text((72, 72), "Hello PyMuPDF. Maria Lopez paid 350 euros.")
doc.save("sample.pdf")
doc.close()

In [7]:
doc = pymupdf.open("sample.pdf")
print(len(doc), "page(s)")
for page in doc:
    print(page.get_text())

1 page(s)
Hello PyMuPDF. Maria Lopez paid 350 euros.



In [8]:
# the same text can go straight into GLiNER
for page in doc:
    for e in model.predict_entities(page.get_text(), labels2, threshold=0.5):
        print(e["text"], "=>", e["label"])

PyMuPDF => organization
Maria Lopez => person
350 euros => amount
